# SESSION A — official peft
### LoRA · BOFT · probe · all of Chapters 1, 3, 4

**Settings:** GPU **T4 ×2** · Internet **ON** · Persistence **Variables and Files**

This notebook is pinned to **official peft**. Its twin, `SESSION B`, is pinned to the **MoRA fork**. They cannot share an environment — `peft-mora` is a fork of peft 0.9.0, which predates BOFT, and installing it overwrites official peft.

**Never change `ENV` in this notebook.** That is the entire reason there are two.

| | Session A (here) | Session B |
|---|---|---|
| peft | official | the fork |
| `--peft` | `lora` `boft` `probe` `dpo` | `mora` `lora` |
| chapters | 1, 3, 4 + part of 2 | part of 2 |

★ **`--peft lora` runs in BOTH**, at the same `--entities` and `--seed`. That is the cross-environment control: if the two LoRA numbers agree, peft version is not a confound.

---

### Known environment facts (measured, not assumed)

* **fp16 + `sdpa`** — fp16 + `eager` returns **NaN** logits on Qwen2.5. Verified by the dtype probe in section 2.
* **One GPU per job.** Two visible → Trainer uses `DataParallel` → autocast never reaches the replicas → `mat1 and mat2 must have the same dtype`.
* **`torchao` must be removed.** Kaggle ships 0.10.0; transformers refuses to import when it is present and below 0.16. Nothing here uses it.
* **`transformers==4.57.6`, pinned.** Bare `pip install -U peft` drags in 5.x, which breaks torchao and drops bitsandbytes. **Session B must pin the same version** or the LoRA control compares two whole stacks instead of isolating peft.
* **BOFT runs with `boft_n_butterfly_factor=1`**, deliberately. Its multi-stage CUDA kernel does not compile against CUDA 12.8. Report this as a limitation.

## 0 · Clone + install

In [ ]:
import socket, urllib.request
for host in ("github.com", "huggingface.co"):
    try:
        print(f"DNS   ok   {host} -> {socket.gethostbyname(host)}")
    except Exception as e:
        print(f"DNS   FAIL {host}: {e}    <-- Settings > Internet > ON")
try:
    print("HTTPS ok   status", urllib.request.urlopen("https://github.com", timeout=10).status)
except Exception as e:
    print("HTTPS FAIL:", e)

In [ ]:
REPO_URL = "https://github.com/lynda-lagh/contribution-.git"
DEST     = "/kaggle/working/repo"

import os, subprocess, sys

def run(cmd):
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode:
        raise SystemExit(f"$ {' '.join(cmd)}\n{r.stdout}{r.stderr}")
    return r.stdout.strip()

if os.path.isdir(f"{DEST}/.git"):
    run(["git", "-C", DEST, "fetch", "--all"])
    run(["git", "-C", DEST, "reset", "--hard", "origin/main"])
    print("updated existing clone")
else:
    run(["git", "clone", "--depth", "1", REPO_URL, DEST])
    print("cloned")

os.chdir(DEST)
if DEST not in sys.path:
    sys.path.insert(0, DEST)

print("HEAD:", run(["git", "log", "-1", "--oneline"]))
print("\n\u2605 Does that hash match your latest push?")

In [ ]:
# ============ SESSION A — DO NOT CHANGE ============
ENV = "official"
PIN_TRANSFORMERS = "4.57.6"     # MUST be identical in Session B
# ===================================================

import json, subprocess, sys

def stack():
    code = ("import inspect, json, peft, transformers; print(json.dumps({"
            "'peft': peft.__version__, 'tf': transformers.__version__,"
            "'mora': 'use_mora' in inspect.signature(peft.LoraConfig.__init__).parameters,"
            "'boft': hasattr(peft, 'BOFTConfig')}))")
    r = subprocess.run([sys.executable, "-c", code], capture_output=True, text=True)
    return json.loads(r.stdout) if r.returncode == 0 else None

s = stack()
if s and s["boft"] and s["tf"] == PIN_TRANSFORMERS:
    print(f"stack correct: peft {s['peft']} / transformers {s['tf']} — skipping install")
else:
    print(f"installing… (have: {s})")
    !pip install -q -r requirements.txt
    !pip install -q peft "transformers=={PIN_TRANSFORMERS}"
    s = stack()

# ★ torchao 0.10.0 ships with Kaggle and makes transformers refuse to import
#   ("only versions above 0.16.0 are supported"), which kills LoRA. Nothing in
#   this pipeline uses it, so remove it rather than chase a compatible version.
!pip uninstall -y -q torchao 2>/dev/null

import peft, transformers
from src.utils.config import peft_env, usable_peft_methods
print(f"\nSESSION A | peft {peft.__version__} | transformers {transformers.__version__}")
print("detected:", peft_env()["peft_env"], "| runnable:", ", ".join(usable_peft_methods()))
assert peft_env()["has_boft"], "BOFT missing — this notebook must use OFFICIAL peft"
assert not peft_env()["has_mora"], "MoRA fork detected — that belongs in Session B"

## 0b · BOFT CUDA kernel patch — ~2 min

peft 0.19.1's `fbd_cuda_kernel.cu` uses the deprecated `Tensor.type()` where current PyTorch
requires `.scalar_type()`, so `nvcc` fails and peft **silently drops `boft_n_butterfly_factor` 2 → 1**.
The butterfly factor is BOFT's structural parameter, so that is a change of *method*, not of speed.

This cell patches the two lines, clears the cached failed build, and **verifies the compile**.
It edits `site-packages`, so it must re-run in every fresh session.

* prints `BUILD OK` → set `boft_n_butterfly_factor: 2` in `configs/base.yaml`
* prints `BUILD STILL FAILS` → keep `1` and report it. Nothing is broken.


In [ ]:
# ============ BOFT CUDA KERNEL PATCH -- run before the smoke test ============
# peft 0.19.1 ships fbd_cuda_kernel.cu written against an OLD PyTorch C++ API.
# Lines 66 / 97 pass `Tensor.type()` (returns at::DeprecatedTypeProperties) to
# AT_DISPATCH_FLOATING_TYPES_AND_HALF, which on current torch wants a
# c10::ScalarType. nvcc therefore fails with:
#     no suitable conversion from "const at::DeprecatedTypeProperties"
#                              to "c10::ScalarType" exists
# peft catches that, warns, and SILENTLY sets boft_n_butterfly_factor -> 1,
# which changes the METHOD, not just its speed.
#
# `.type()` -> `.scalar_type()` is the modern spelling of the same thing.
# This patches site-packages, so it must re-run in EVERY fresh Kaggle session.
import pathlib, shutil, subprocess, sys, peft

kdir = pathlib.Path(peft.__file__).parent / "tuners" / "boft" / "fbd"
cu   = kdir / "fbd_cuda_kernel.cu"

src = cu.read_text()
new = (src.replace("input.type()",       "input.scalar_type()")
          .replace("grad_output.type()", "grad_output.scalar_type()"))

if new != src:
    cu.with_suffix(".cu.orig").write_text(src)      # keep the original
    cu.write_text(new)
    print(f"patched  {cu}")
else:
    print("already patched (or upstream fixed it) -- no change")

# the failed build is cached; it must go or torch reuses the failure
shutil.rmtree("/root/.cache/torch_extensions", ignore_errors=True)

# ---- verify: actually compile it, in a subprocess so a hard crash can't
# ---- take the notebook kernel with it. First build takes 1-2 min.
print("\nbuilding fbd_cuda (1-2 min on first run) ...")
r = subprocess.run(
    [sys.executable, "-c",
     "import warnings; warnings.simplefilter('ignore');"
     "from peft.tuners.boft.layer import get_fbd_cuda;"
     "print('BUILD_OK' if get_fbd_cuda() is not None else 'BUILD_FAILED')"],
    capture_output=True, text=True, timeout=900)
out = r.stdout + r.stderr

if "BUILD_OK" in out:
    print("\n  BOFT CUDA extension BUILT.")
    print("  -> set boft_n_butterfly_factor: 2 in configs/base.yaml")
    print("     (BOFT now runs as published, two butterfly stages)")
else:
    print("\n  BUILD STILL FAILS -- there is a second incompatibility.")
    print("  -> KEEP boft_n_butterfly_factor: 1 in configs/base.yaml and")
    print("     report that BOFT ran with a single butterfly stage.")
    print("     Nothing is broken; this is the documented fallback.")
    print("\n----- compiler output (last 40 lines) -----")
    print("\n".join(out.strip().splitlines()[-40:]))


## 1 · Data — downloaded, not uploaded

In [ ]:
!python -m scripts.fetch_data --datasets WN11 FB13

FETCH_BIG = False        # True before Chapter 2 (YAGO3-10 is ~1M triples)
if FETCH_BIG:
    !python -m scripts.fetch_data --datasets WN18RR YAGO3-10

## 2 · Smoke test — ~5 min

Expect **MoRA to fail** here; it lives in Session B. Everything else must pass.

⚠️ `train_loss` of exactly `0.0000` is a failure, not a pass.

In [ ]:
!CUDA_VISIBLE_DEVICES=0 python -m scripts.smoke_test

## 3 · Build instruction data

10,000 triples → ~20,000 instances. Stratified by relation, min 10 each. No GPU needed.

`--anonymise` is the contamination control — WN11 and FB13 date from 2013 and predate every LLM's training cutoff.

In [ ]:
!python -m src.data.build_instructions --dataset WN11 --n_triples 10000 --seed 42
!python -m src.data.build_instructions --dataset WN11 --n_triples 10000 --seed 42 --anonymise

## 4 · Chapter 1 — format vs knowledge

Two independent training runs → one per T4.

Binary task → chance = 50%, which is what makes KG-LLM's untuned 21.1 / 9.1 unambiguous: a model cannot be wrong about facts five times more reliably than a coin.

> Evaluation uses a fixed 2,000-item subset of WN11's 21,088 test rows — identical items across conditions, which is what makes the paired tests valid. Not directly comparable to KG-LLM's full-test figures; say so in the paper.

In [ ]:
import subprocess

def pair(a, b):
    """Two INDEPENDENT jobs, one per T4. Never DataParallel."""
    pa = subprocess.Popen(f"CUDA_VISIBLE_DEVICES=0 {a}", shell=True)
    pb = subprocess.Popen(f"CUDA_VISIBLE_DEVICES=1 {b}", shell=True)
    return pa.wait(), pb.wait()

pair("python -m chapters.ch1_diagnostic.run --dataset WN11",
     "python -m chapters.ch1_diagnostic.run --dataset WN11 --anonymise")

In [ ]:
# four parsers on identical outputs + SMI as a second, independent instrument
!CUDA_VISIBLE_DEVICES=0 python -m chapters.ch1_diagnostic.analyse --dataset WN11 --smi

## 5 · Chapter 2 — the arms that need official peft

Set `FETCH_BIG = True` in section 1 first.

`probe` = frozen linear classifier on hidden states, **no training**. It is the control that makes a flat MoRA result interpretable rather than ambiguous: if every method merely matches a probe, no method installed knowledge.

In [ ]:
BASE = "python -m chapters.ch2_adaptation.run --dataset YAGO3-10 --triples 10000"

pair(f"{BASE} --peft boft  --entities 123182",
     f"{BASE} --peft probe --entities 123182")

In [ ]:
# \u2605 THE CROSS-ENVIRONMENT CONTROL.
# Session B must run this EXACT command. Same --entities, same --seed.
!CUDA_VISIBLE_DEVICES=0 python -m chapters.ch2_adaptation.run \
        --dataset YAGO3-10 --triples 10000 --peft lora --entities 123182 --seed 42

## 6 · Chapter 3 — the conditioning ladder

Routing + faithfulness cost **no training**. Check the `rich` band in `--analyse`: if it is ~0%, the ∅ branch never fires and a 0% skip rate is a data problem, not a result.

⚠️ Relation descriptions are currently template-derived. For the real run they must be LLM-generated, or L1 injects boilerplate and the first rung measures nothing.

In [ ]:
!python -m chapters.ch3_conditioning.run --dataset YAGO3-10 --analyse

In [ ]:
for LEVEL in ["L0", "L1", "L2", "L3"]:          # L4 is the first thing to cut
    !CUDA_VISIBLE_DEVICES=0 python -m chapters.ch3_conditioning.run --dataset YAGO3-10 --level {LEVEL} --train

## 7 · Chapter 4 — measurement

★ **ZERO training.** Inference over checkpoints from Chapters 2–3. This chapter holds the verified gaps: calibration (2 of 188), abstention (0), explanation faithfulness (0).

In [ ]:
ADAPTER = "checkpoints/ch2-lora-E123182-T10000-s42"   # or the Phase-1 winner
!CUDA_VISIBLE_DEVICES=0 python -m chapters.ch4_measurement.run --adapter {ADAPTER} --dataset YAGO3-10 --limit 2000

---
### Session A checklist

* [ ] commit hash matches your latest push
* [ ] `transformers 4.57.6` — **identical in Session B**
* [ ] `torchao` absent
* [ ] BOFT patch cell run; `boft_n_butterfly_factor` in `configs/base.yaml` matches its verdict
* [ ] smoke test: only MoRA failed
* [ ] `--peft lora --entities 123182 --seed 42` run here **and** in Session B
* [ ] `results_sessionA.zip` downloaded


In [ ]:
!zip -qr /kaggle/working/results_sessionA.zip results/
!du -sh /kaggle/working/results_sessionA.zip
!ls results/ | head -20

---
### Session A checklist

* [ ] commit hash matches your latest push
* [ ] `transformers 4.57.6` — **identical in Session B**
* [ ] `torchao` absent
* [ ] smoke test: only MoRA failed
* [ ] `--peft lora --entities 123182 --seed 42` run here **and** in Session B
* [ ] `results_sessionA.zip` downloaded